# 03 — Dataset Formation for the Embedding Model

Prepares the input for the **note-embedding strategy**. Unlike `02` (which builds cumulative
`NOTE_TEXT` per snapshot for the LLM), here every note is kept **on its own, as written**: one
row per note, cleaned of HTML only. Each note is embedded exactly once in `04`, and the claim's
running history is built afterwards from those per-note vectors in `05`.

**Rules**
1. One record per note: `note_id`, `claim_num`, `date`, `text` (HTML stripped, whitespace normalized).
2. All notes are embedded. A note's embedding doesn't depend on any target, so one embedding per
   note is reused by every model; target-specific cutoffs are applied later, in `05`.
3. **No split here.** The train/val/holdout split is done once, in `02`. Notebook `05` takes each
   snapshot row (with its split) straight from `02`'s datasets, so both strategies use the same split.
4. Notes are sharded into 20 JSON-Lines files (round-robin) so the embedding endpoint can spread
   the work evenly.

Output: `s3://bi-severity-capstone/adjuster-notes/embedding/input/notes_XXX.jsonl`

In [1]:
import glob, os, re, html, json, time
import numpy as np
import pandas as pd
import boto3

import sys; sys.path.insert(0, '..')   # repo root, so config.py can be imported
from config import NOTES_DIR, S3_BUCKET, TMP_DIR, EMBEDDING_PREFIX as S3_PREFIX   # paths: config.py
N_SHARDS   = 20
SEED       = 42


## Step 1 — load and clean every note (single note, kept as written)

Only HTML is removed (`<br>` becomes a newline, other tags dropped, entities unescaped, repeated
spaces collapsed). Shorthand, casing and abbreviations are left untouched, since that's exactly
what the embedding model has to read.

In [5]:
BR, TAG, WS = re.compile(r'<br\s*/?>', re.I), re.compile(r'<[^>]+>'), re.compile(r'[ \t]+')

def clean_note(t):
    if pd.isna(t):
        return ''
    t = html.unescape(str(t))
    t = BR.sub('\n', t)
    t = TAG.sub('', t)
    t = WS.sub(' ', t)
    return t.strip()


note_files = sorted(glob.glob(os.path.join(NOTES_DIR, 'batch_*.csv')))  # reads local data
notes = pd.concat([pd.read_csv(f, usecols=['CLAIM_NUM', 'DATE', 'TEXT_NOTE'], dtype={'CLAIM_NUM': str})  # reads local data
                   for f in note_files], ignore_index=True)
notes['DATE'] = pd.to_datetime(notes['DATE'], errors='coerce').dt.strftime('%Y-%m-%d')
notes['TEXT'] = notes['TEXT_NOTE'].map(clean_note)

print(f'{len(notes):,} notes across {notes.CLAIM_NUM.nunique():,} distinct claims')

4,645,819 notes across 138,233 distinct claims

## Step 2 — shard into JSON-Lines (round-robin) and upload

In [6]:
LOCAL_DIR = str(TMP_DIR / 'emb_input')
os.makedirs(LOCAL_DIR, exist_ok=True)
writers = [open(os.path.join(LOCAL_DIR, f'notes_{i:03d}.jsonl'), 'w') for i in range(N_SHARDS)]
shard_counts = [0] * N_SHARDS

for seq, (claim, date, text) in enumerate(zip(notes.CLAIM_NUM, notes.DATE, notes.TEXT)):
    rec = {'note_id': f'{claim}_{seq}', 'claim_num': claim, 'date': date, 'text': text}
    writers[seq % N_SHARDS].write(json.dumps(rec) + '\n')
    shard_counts[seq % N_SHARDS] += 1
for w in writers:
    w.close()

s3 = boto3.client('s3')
for i in range(N_SHARDS):
    name = f'notes_{i:03d}.jsonl'
    s3.upload_file(os.path.join(LOCAL_DIR, name), S3_BUCKET, f'{S3_PREFIX}/input/{name}')  # S3 write
    print(f'  {name}: {shard_counts[i]:,} notes')

print(f'\nDONE: {sum(shard_counts):,} notes across {N_SHARDS} shards uploaded to '
      f's3://{S3_BUCKET}/{S3_PREFIX}/input/')

  notes_000.jsonl: 232,291 notes
  notes_001.jsonl: 232,291 notes
  notes_002.jsonl: 232,291 notes
  notes_003.jsonl: 232,291 notes
  notes_004.jsonl: 232,291 notes
  notes_005.jsonl: 232,291 notes
  notes_006.jsonl: 232,291 notes
  notes_007.jsonl: 232,291 notes
  notes_008.jsonl: 232,291 notes
  notes_009.jsonl: 232,291 notes
  notes_010.jsonl: 232,291 notes
  notes_011.jsonl: 232,291 notes
  notes_012.jsonl: 232,291 notes
  notes_013.jsonl: 232,291 notes
  notes_014.jsonl: 232,291 notes
  notes_015.jsonl: 232,291 notes
  notes_016.jsonl: 232,291 notes
  notes_017.jsonl: 232,291 notes
  notes_018.jsonl: 232,291 notes
  notes_019.jsonl: 232,290 notes

DONE: 4,645,819 notes across 20 shards uploaded to s3://bi-severity-capstone/adjuster-notes/embedding/input/